## 数据处理版本v2: 
### 2025-0907

对csv手动进行了一些处理：

AB_C
B_CD
手动修改【DADA_MN（重-轻）/ 4U6H_AB_E【A重链-B轻链】/3NPS_A_BC【B-重链-C轻链】】
都改成：前轻后重；
使用前轻后重； 单独的为抗原；
HL如果有，单独的为抗原。 数据标注完成

4NM8_ABCDEF_HL，保留B
5DWU_HL_AB，保留B链作为抗原，删除一条突变点再A上的样本数据。
4KRP_A_B其中B是纳米抗体，所以在tsv中修改为4KRP_A_CB，将B处理为重链，并且在seqs中补入C=“”

In [1]:

import tarfile
tar = tarfile.open(r"/root/private_data/luog/Data_AbAg/skempi/SKEMPI2_PDBs.tgz", 'r:gz')
name_list = [m.name for m in tar.getmembers()]
mapping_files = {name.split('/')[-1].split('.')[0]: name for name in name_list if name.endswith('.mapping')}  

In [4]:
import warnings
from Bio import PDB
import numpy as np
import json
from tqdm import tqdm
from Bio.Data.PDBData import protein_letters_3to1_extended 

def save_to_json(data, json_file):
    with open(json_file, 'w', encoding='utf-8') as jsonfile:
        json.dump(data, jsonfile, ensure_ascii=False, indent=None)


def read_mapping_file(file):
    extended_andUNK = protein_letters_3to1_extended
    extended_andUNK["UNK"] = "X"  # 如果为UNK则修改为X，最终修改掉突变之后删除前面的X
    file = file.split("\n")
    chains = {}
    for line in file:
        parts = line.split()
        if len(parts) == 0:
            continue
        if len(parts) == 3:
            parts = [parts[0], parts[1][0],parts[1][1:] ,parts[2]]
        
        if parts[1] not in chains.keys():
            chains[parts[1]] = extended_andUNK[parts[0]]
        else:
            chains[parts[1]] = chains[parts[1]] + extended_andUNK[parts[0]]
    return chains


result=[]
for k,v in tqdm(mapping_files.items()):
    try:
        f = tar.extractfile(v)
        content = f.read().decode('utf-8')
        seqs = read_mapping_file(content)
        f.close()
        result.append({"pdb":k,"seqs":seqs})
    except:
        print(k,v)
        continue
save_to_json(result,"/root/private_data/luog/Data_AbAg/skempi/result/skempi_seqsinfo.json")

  4%|▍         | 15/346 [00:00<00:15, 21.11it/s]

 PDBs/._5DWU.mapping


100%|██████████| 346/346 [00:01<00:00, 203.95it/s]


In [ ]:
from Bio.Data.PDBData import protein_letters_3to1_extended 
import tarfile
import json
import os
import json
import random
import string
from tqdm import tqdm
import copy
import warnings
from Bio import PDB
import numpy as np
import json
from tqdm import tqdm

def save_to_json(data, json_file):
    with open(json_file, 'w', encoding='utf-8') as jsonfile:
        json.dump(data, jsonfile, ensure_ascii=False, indent=None)


def fix_col(i):
    # 如果给了突变信息，则修改序列，再做轻/重/抗原链赋值
    seqs = i["seqs"]
    if i["mutation"]!="wt_type":
        Mutation_Point = i["mutation"].split(",")
        for p in Mutation_Point:
            index = int(p[2:-1])-1
            seq = seqs[p[1]] # 获取原始序列
            if seq[index]==p[0]:
                seqs[p[1]] =  seq[:index] + p[-1] + seq[index+1:] # 赋值
                # print("修改序列")
            else:
                print(i["pdb"],p,seq[index-1],p[0],"修改失败")

    chains_info = i["pdb"].split("_",1)[1]
    if ("HL" in chains_info) or ("LH" in chains_info):
        chains_other = chains_info.replace("_","").replace("HL","").replace("LH","")
        i["X"] = seqs[chains_other]
        i["P"] = ""
        i["H"] = seqs["H"]
        i["L"] = seqs["L"]
        i["seqs"] = chains_other+"HL"
    else:
        chains_info = chains_info.split("_")
        for j in chains_info:
            if len(j) == 2:
                # 前轻后重
                i["L"] = seqs[j[0]]
                i["H"] = seqs[j[1]]
            else:
                i["X"] = seqs[j]

        i["seqs"] = "XXX"
    return i

def generate_id(length=5):
    return ''.join(random.choices(string.ascii_uppercase, k=length))

def skermpi(csv_data,save_dir):
    with open("/root/private_data/luog/Data_AbAg/skempi/result/skempi_seqsinfo.json", 'r', encoding='utf-8') as f:
        data_seqs = json.load(f)
        seqs_dict={}
        for i in data_seqs:
            seqs_dict[i["pdb"]]=i["seqs"]
        
    data_wt=[] 
    data_mt=[] 
    pdb_set=[]


    with open(csv_data, 'r', encoding='utf-8') as f:
        data = f.readlines() 
        for i in tqdm(data[1:]):
            data_i = i.strip().split(";")
            pdb_name = data_i[0].split("_")[0].upper()
        # try:
            if "AB/AG" in data_i[4]:
                # 突变型放入数据
                col_mt = fix_col({
                        "pdb":data_i[0],
                        "seqs": copy.deepcopy(seqs_dict[pdb_name]),
                        "mutation":data_i[2],
                        "kd":-np.log10(float(data_i[7])) if data_i[7]!="" else 0.0,
                        "kon":-np.log10(float(data_i[15])) if data_i[15]!="" else 0.0,
                        "koff":-np.log10(float(data_i[19])) if data_i[19]!="" else 0.0,
                        "Hold_out_type":data_i[4],
                    })
                col_mt["pdb"] = pdb_name+"_"+generate_id(5)
                data_mt.append(col_mt)
                # 野生型放入数据
                if pdb_name not in pdb_set:
                    pdb_set.append(pdb_name) 
                    col_wt = fix_col({
                        "pdb":data_i[0],
                        "seqs": copy.deepcopy(seqs_dict[pdb_name]),
                        "mutation":"wt_type",
                        "kd":-np.log10(float(data_i[9])) if data_i[9]!="" else 0.0,
                        "kon":-np.log10(float(data_i[17])) if data_i[17]!="" else 0.0,
                        "koff":-np.log10(float(data_i[21])) if data_i[21]!="" else 0.0,
                        "Hold_out_type":data_i[4],
                    })
                    col_wt["pdb"] = pdb_name+"_"+generate_id(5)
                    data_wt.append(col_wt)

            
    data_all = data_wt + data_mt

    save_to_json(data_all,os.path.join(save_dir,"skempi_AbAg_"+str(len(data_all))+".json"))
    data_koff = []
    for i in data_all:
        if i["koff"] > 0.0:
            data_koff.append(i)
    save_to_json(data_koff,os.path.join(save_dir,"skempi_AbAgAoff_"+str(len(data_koff))+".json"))
    return data_all

random.seed(42)
csv_data="/root/private_data/luog/Data_AbAg/skempi/skempi_v2_change.csv"
save_dir="/root/private_data/luog/Data_AbAg/skempi/result"
data_all = skermpi(csv_data,save_dir)
print("一共",len(data_all),"条数据")


100%|██████████| 7037/7037 [00:00<00:00, 7775.12it/s] 

一共 1217 条数据
